In [1]:
import xarray as xr
import numpy as np

In [2]:
path = "fire_pred_dataset.nc"
#environmental_data/data_0.nc
#fire_data/copernicus/20240101-C3S-L3-FRP-SLSTR-P1D-0.1deg-S3A-daytime-fv1.2.nc
#fire_data/SUOMI_VIIRS_C2_Global_VNP14IMGTDL_NRT_2025055.txt

In [3]:
ds = xr.open_dataset(path)
ds2 = xr.open_dataset("combined_data/jan2025.nc")

In [4]:
print(list(ds.data_vars))

print(f"Time range: {ds.time.values.min()} to {ds.time.values.max()}")
print(f"Latitude range: {ds.lat.values.min():.4f} to {ds.lat.values.max():.4f}")
print(f"Longitude range: {ds.lon.values.min():.4f} to {ds.lon.values.max():.4f}")

['frp', 'pr', 'rmax', 'rmin', 'sph', 'srad', 'th', 'tmmn', 'tmmx', 'vs', 'erc', 'eto', 'bi', 'fm100', 'fm1000', 'etr', 'vpd', 'LAI_AVE']
Time range: 2025-01-01T00:00:00.000000000 to 2025-01-31T00:00:00.000000000
Latitude range: 34.0085 to 34.7126
Longitude range: -119.0846 to -117.5855


In [ ]:
# Cropping

# Bound
new_min_lat = 34.0530
new_min_lon = -118.9530

delta_lat = ds.lat.max().item() - ds.lat.min().item()
delta_lon = ds.lon.max().item() - ds.lon.min().item()

new_max_lat = new_min_lat + delta_lat
new_max_lon = new_min_lon + delta_lon

is_lat_descending = ds.lat.values[0] > ds.lat.values[-1]

# Crop
lat_slice = slice(new_max_lat, new_min_lat) if is_lat_descending else slice(new_min_lat, new_max_lat)
lon_slice = slice(new_min_lon, new_max_lon)

cropped_ds = ds.sel(lat=lat_slice, lon=lon_slice)

cropped_ds.to_netcdf("cropped_fire_pred_dataset.nc")

print("done.")

done.


In [6]:
ds3 = xr.open_dataset("cropped_fire_pred_dataset.nc")

In [7]:
print(f"Time range: {ds3.time.values.min()} to {ds3.time.values.max()}")
print(f"Latitude range: {ds3.lat.values.min():.4f} to {ds3.lat.values.max():.4f}")
print(f"Longitude range: {ds3.lon.values.min():.4f} to {ds3.lon.values.max():.4f}")

Time range: 2025-01-01T00:00:00.000000000 to 2025-01-31T00:00:00.000000000
Latitude range: 34.0557 to 34.7126
Longitude range: -118.9499 to -117.5855


In [8]:
frp = ds3['frp']

total_elements = frp.size

num_nans = np.isnan(frp).sum().item()
num_zeros = ((frp == 0) & (~np.isnan(frp))).sum().item()
percentage = ((num_nans + num_zeros) / total_elements) * 100

# Output
print(f"Total elements in frp: {total_elements:,}")
print(f"NaN values: {num_nans:,}")
print(f"Zero values: {num_zeros:,}")
print(f"NaN + Zero count: {num_nans + num_zeros:,}")
print(f"Percentage of NaN or Zero: {percentage:.2f}%")

Total elements in frp: 2,466,856
NaN values: 2,457,794
Zero values: 0
NaN + Zero count: 2,457,794
Percentage of NaN or Zero: 99.63%


In [9]:
ds4 = xr.open_dataset("spread_fire_data.nc")

In [10]:
frp = ds4['frp']

total_elements = frp.size

num_nans = np.isnan(frp).sum().item()
num_zeros = ((frp == 0) & (~np.isnan(frp))).sum().item()
percentage = ((num_nans + num_zeros) / total_elements) * 100

# Output
print(f"Total elements in frp: {total_elements:,}")
print(f"NaN values: {num_nans:,}")
print(f"Zero values: {num_zeros:,}")
print(f"NaN + Zero count: {num_nans + num_zeros:,}")
print(f"Percentage of NaN or Zero: {percentage:.2f}%")

Total elements in frp: 2,466,856
NaN values: 0
Zero values: 2,269,853
NaN + Zero count: 2,269,853
Percentage of NaN or Zero: 92.01%


In [12]:
from pyproj import Geod
lats = ds3["lat"].values
lons = ds3["lon"].values

# Initialize geodetic calculator
geod = Geod(ellps="WGS84")

# PIck 2
lat1, lat2 = lats[0], lats[1]
lon1, lon2 = lons[0], lons[1]

dlat_deg = lat2 - lat1
dlon_deg = lon2 - lon1

print(f"Latitude degree step : {dlat_deg:.6f}°")
print(f"Longitude degree step: {dlon_deg:.6f}°")

# Compute h/w in meters
_, _, height_m = geod.inv(lon1, lat1, lon1, lat2)
_, _, width_m  = geod.inv(lon1, lat1, lon2, lat1)

print(f"Approximate cell height (N-S): {height_m:.1f} m")
print(f"Approximate cell width  (E-W): {width_m:.1f} m")

Latitude degree step : -0.003369°
Longitude degree step: 0.003369°
Approximate cell height (N-S): 373.7 m
Approximate cell width  (E-W): 308.6 m
